In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json



In [47]:
folder_path = "/Users/prafull/Downloads/results_by_game/"

method_folders = [f for f in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, f))]

data = {}

for method_folder in method_folders:
    method_path = os.path.join(folder_path, method_folder)
    game_id = 0
    for game_folder in sorted(os.listdir(method_path)):
        game_name = f"game_{game_id:04d}"
        if os.path.isdir(os.path.join(method_path, game_folder)):
            for user_folder in sorted(os.listdir(os.path.join(method_path, game_folder))):
                if os.path.isdir(os.path.join(method_path, game_folder, user_folder)):
                    files = os.listdir(os.path.join(method_path, game_folder, user_folder))
                    json_files = [f for f in files if f.endswith(".json")]
                    user_ratings = json.load(open(os.path.join(method_path, game_folder, user_folder, json_files[0])))
                    if game_name not in data:
                        data[game_name] = {}
                    if method_folder not in data[game_name]:
                        data[game_name][method_folder] = {}
                    data[game_name][method_folder][user_folder] = user_ratings["fun"], user_ratings["playability"], user_ratings["explanation"]
        game_id += 1


{'method_vlm_play_unguided_Single_Prompt_Basic': {'user_5f7529755aaa0e1a6e804640': (0, 0, 'This game was completely unplayable.  Might have been a glitch or something.'), 'user_614e50084109bcceada6945c': (0, 0, 'The game didn’t do anything except display a black screen of blue arrows in vertical lines. The controls didn’t trigger or start anything.'), 'user_664439645a9a29be43d5bf5a': (0, 0, 'game doesnt work'), 'user_67d86290a2be1370fc1e4083': (73, 86, 'na'), 'user_67e03de86567af68e15f8b33': (0, 0, "It was just a black screen with arrows on it, I couldn't play it.")}, 'method_Single_Prompt_Basic': {'user_5f7529755aaa0e1a6e804640': (50, 50, 'too hard and kinda mid'), 'user_664439645a9a29be43d5bf5a': (0, 0, 'boring'), 'user_66a2812744e8026f80ef0879': (60, 50, "It's not very easy with my circle always changing size. I'm not sure why it keeps ending."), 'user_67d86290a2be1370fc1e4083': (95, 91, 'The game is engaging and entertaining, with a good balance of challenge and reward, though it c

In [49]:

# Create a dictionary to store all user data by game and method
all_user_data = {}

# Get all unique users across all games and methods
all_users = set()
for game_methods in data.values():
    for users in game_methods.values():
        all_users.update(users.keys())

# Create a multi-level dictionary structure
for game_name, methods in data.items():
    all_user_data[game_name] = {}
    for method_name in method_folders:
        all_user_data[game_name][method_name] = {}
        if method_name in methods:
            # Add existing user data
            for user, (fun, playability, explanation) in methods[method_name].items():
                all_user_data[game_name][method_name][user] = (fun, playability, explanation)

# Convert to DataFrame with MultiIndex
rows = []
indices = []

for game_name, methods in all_user_data.items():
    for method_name, users in methods.items():
        for user, (fun, playability, explanation) in users.items():
            row_data = {
                'fun': fun,
                'playability': playability,
                'explanations': explanation  # Keep full explanation text
            }
            rows.append(row_data)
            indices.append((game_name, method_name, user))

# Create MultiIndex for rows (game, method, user)
multi_index = pd.MultiIndex.from_tuples(indices, names=['game', 'method', 'user'])

# Create DataFrame with hierarchical index
dataframe = pd.DataFrame(rows, index=multi_index)

In [51]:
for game_name, methods in all_user_data.items():
    for method_name, users in methods.items():
        for user in users:
            print(f"({game_name}, {method_name}) {user}, Fun: {users[user][0]}, Playability: {users[user][1]}, Explanation: {users[user][2]}")


(game_0001, method_vlm_play_unguided_Single_Prompt_Basic) user_5f7529755aaa0e1a6e804640, Fun: 0, Playability: 0, Explanation: This game was completely unplayable.  Might have been a glitch or something.
(game_0001, method_vlm_play_unguided_Single_Prompt_Basic) user_614e50084109bcceada6945c, Fun: 0, Playability: 0, Explanation: The game didn’t do anything except display a black screen of blue arrows in vertical lines. The controls didn’t trigger or start anything.
(game_0001, method_vlm_play_unguided_Single_Prompt_Basic) user_664439645a9a29be43d5bf5a, Fun: 0, Playability: 0, Explanation: game doesnt work
(game_0001, method_vlm_play_unguided_Single_Prompt_Basic) user_67d86290a2be1370fc1e4083, Fun: 73, Playability: 86, Explanation: na
(game_0001, method_vlm_play_unguided_Single_Prompt_Basic) user_67e03de86567af68e15f8b33, Fun: 0, Playability: 0, Explanation: It was just a black screen with arrows on it, I couldn't play it.
(game_0001, method_Single_Prompt_Basic) user_5f7529755aaa0e1a6e804

In [ ]:
data["game_0002"]

In [44]:
def display_game_ratings(game_id):
    """
    Display ratings for a specific game with colored backgrounds for better readability.
    
    Args:
        game_id (str): The game ID to display (e.g., 'game_0002')
    """
    import pandas as pd
    from IPython.display import display, HTML
    
    if game_id not in data:
        print(f"Game {game_id} not found in the dataset.")
        return
    
    game_data = data[game_id]
    
    # Create HTML for better visualization
    html_content = f"""
    <h2 style="text-align:center; background-color:#f0f0f0; padding:10px; border-radius:5px; color:black;">
        Ratings for {game_id}
    </h2>
    """
    
    for method, users in game_data.items():
        html_content += f"""
        <div style="margin:15px 0; padding:10px; background-color:#e6f7ff; border-radius:5px;">
            <h3 style="color:black;">{method}</h3>
            <table style="width:100%; border-collapse:collapse;">
                <tr style="background-color:#0066cc;">
                    <th style="padding:8px; text-align:left; border:1px solid #ddd; color:black;">User</th>
                    <th style="padding:8px; text-align:center; border:1px solid #ddd; color:black;">Fun</th>
                    <th style="padding:8px; text-align:center; border:1px solid #ddd; color:black;">Playability</th>
                    <th style="padding:8px; text-align:left; border:1px solid #ddd; color:black;">Explanation</th>
                </tr>
        """
        
        for user, (fun, playability, explanation) in users.items():
            # Color coding for ratings
            fun_color = get_rating_color(fun)
            playability_color = get_rating_color(playability)
            
            html_content += f"""
                <tr style="background-color:#f9f9f9;">
                    <td style="padding:8px; border:1px solid #ddd; color:black;">{user}</td>
                    <td style="padding:8px; text-align:center; border:1px solid #ddd; background-color:{fun_color}; color:black;">{fun}</td>
                    <td style="padding:8px; text-align:center; border:1px solid #ddd; background-color:{playability_color}; color:black;">{playability}</td>
                    <td style="padding:8px; border:1px solid #ddd; color:black;">{explanation}</td>
                </tr>
            """
        
        html_content += """
            </table>
        </div>
        """
    
    display(HTML(html_content))

def get_rating_color(rating):
    """Return a color based on the rating value."""
    if rating >= 80:
        return "#d4edda"  # Green for high ratings
    elif rating >= 50:
        return "#fff3cd"  # Yellow for medium ratings
    else:
        return "#f8d7da"  # Red for low ratings

# Example usage
display_game_ratings("game_0005")



User,Fun,Playability,Explanation
user_5f7529755aaa0e1a6e804640,17,8,Still can't figure this out
user_614e50084109bcceada6945c,0,0,Frustrating. Controls are inaccurate - the space bar doesn’t jump when I press it.
user_67d86290a2be1370fc1e4083,87,73,"The use of vibrant, dynamic colors adds to the excitement—such as warm tones during action scenes and cool, relaxing hues in puzzle or rest areas. These color changes help maintain a stimulating, immersive experience throughout gameplay."
user_67e03de86567af68e15f8b33,0,0,Again my jump button did not work and my ball started flashing yellow.
User,Fun,Playability,Explanation
user_5f7529755aaa0e1a6e804640,70,63,This game was decent had a decent gameplay but the controls were to fast
user_664439645a9a29be43d5bf5a,20,8,controls still difficult
user_66a2812744e8026f80ef0879,39,9,The controls were very difficult. The sub just zoomed around too fast. I didn't feel like I had much control over it.


{'method_vlm_play_unguided_Single_Prompt_Basic': {'user_5f7529755aaa0e1a6e804640': (0, 0, 'This game was completely unplayable.  Might have been a glitch or something.'), 'user_614e50084109bcceada6945c': (0, 0, 'The game didn’t do anything except display a black screen of blue arrows in vertical lines. The controls didn’t trigger or start anything.'), 'user_664439645a9a29be43d5bf5a': (0, 0, 'game doesnt work'), 'user_67d86290a2be1370fc1e4083': (73, 86, 'na'), 'user_67e03de86567af68e15f8b33': (0, 0, "It was just a black screen with arrows on it, I couldn't play it.")}, 'method_Single_Prompt_Basic': {'user_5f7529755aaa0e1a6e804640': (50, 50, 'too hard and kinda mid'), 'user_664439645a9a29be43d5bf5a': (0, 0, 'boring'), 'user_66a2812744e8026f80ef0879': (60, 50, "It's not very easy with my circle always changing size. I'm not sure why it keeps ending."), 'user_67d86290a2be1370fc1e4083': (95, 91, 'The game is engaging and entertaining, with a good balance of challenge and reward, though it c